# 🔋 Running out of Plutonium
## Where a Language Model Goes Blind

**Section 4** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### 🖥️ This is the **local** version.
> It runs entirely on your own GPU.
> The other version is **`03_plutonium_api.ipynb`**.
>
> ⚠️ **These two notebooks reach genuinely different conclusions**, and that is
> not a mistake. Which model you run changes how this failure looks. We will be
> explicit about it.

---

**This notebook continues directly from Section 3.** There we took a vague,
hedging model and, with one sentence of English, turned it into something that
answers like a confident 25-year appraiser.

That was the good news. Here is the bill.

Doc Brown could build a time machine, but he still needed plutonium. Without
fuel, the DeLorean is a stainless steel car with gull-wing doors. A language
model has the same problem, and it has it in three directions at once:

| It cannot see | Because |
|---|---|
| 🕐 **Right now** | It has no clock and no internet. It was trained once and frozen. |
| 🏛️ **Your CAMA system** | It has never seen your parcels, your values, or your sales. |
| 📄 **Your office's policies** | Your manual was not in its training data. Nobody's was. |

---

### What you need

| | |
|---|---|
| ⏱️ | About 4 minutes |
| 🎮 | A GPU with ~5 GB free |
| 🏘️ | The Hill Valley County data (this notebook builds it if missing) |

---

# 🚦 LIVE DEMO PATH

## Setup

The same `ask()` helper from Section 3, and the same expert persona we wrote
there. **This is the only cell that differs between the two versions.**

In [1]:
import torch
from delorean import data, llm, viz

model, tokenizer = llm.load_local("qwen")


def ask(system: str, question: str, max_tokens: int = 130) -> str:
    """Send one system prompt and one question. Get the answer back."""
    messages = []
    if system.strip():
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    text = llm.chat_prompt(tokenizer, messages)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


data.check()   # make sure Hill Valley County exists on this machine

Loading Qwen3.5 4B onto NVIDIA GeForce RTX 3080 ...


W0816 18:33:23.756000 21032 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Ready. Using 2.95 GB of 10.0 GB (4-bit NF4).


In [2]:
# The two prompts this whole notebook turns on.
PLAIN = "You are a helpful assistant."

# This is the persona we wrote in Section 3, barely changed. It is a completely
# reasonable thing for an office to deploy -- which is exactly the problem.
EXPERT = (
    "You are a senior property appraiser with 25 years of experience in the "
    "Hill Valley County assessor's office. You know the county's parcels, "
    "policies, and sales history thoroughly. Answer confidently and concisely."
)

---

## Act 1 — 🌦️ Ask it about right now

The simplest possible question a model cannot answer.

In [3]:
print(ask(PLAIN, "What is the weather in Calgary right now?"))

I don't have access to real-time data, so I can't tell you the current weather in Calgary.

To get the most accurate and up-to-date information, I recommend:
*   Checking a trusted weather website like **Environment Canada** (weather.gc.ca).
*   Using a search engine like Google.
*   Checking a weather app on your phone.


> ### 💡 Notice that it behaved **well**.
>
> It did not invent a temperature. It told you it cannot know and pointed you
> somewhere useful. That is exactly what you want.
>
> **And it is still a failure.** A polite refusal gets an appraiser precisely as
> far as a wrong answer does: nowhere. The model is not broken — it is *blind*,
> and being honest about blindness does not restore sight.

Hold onto this. In **Section 6** we hand the model its first real tool, and
questions it flatly refused here start coming back answered.

---

## Act 2 — 🏛️ Now ask it about your county

Here is a real question from a real workflow. A taxpayer wants to split a
parcel, and you need to know the minimum lot size.

We ask it **twice**: once with no persona, and once with the confident expert
persona from Section 3. Same question. Same model. Same weights.

In [4]:
QUESTION = ("In Hill Valley County, what is the minimum lot size to split an "
            "R-1 parcel?")

plain_answer = ask(PLAIN, QUESTION)
expert_answer = ask(EXPERT, QUESTION)

viz.show_html(viz.persona_card_html(
    "🤖 No persona", plain_answer, accent=viz.PERSONA_COLORS[3]))
viz.show_html(viz.persona_card_html(
    "🏛️ The Section 3 expert persona", expert_answer, accent=viz.PERSONA_COLORS[1]))

> ### ⚠️ Read those two again.
>
> With no persona, the model hedges — it points out that it does not have this
> county's ordinances.
>
> With the persona from Section 3, it produces **a specific number, attributed
> to a specific ordinance, with no hedging at all.** It sounds exactly like the
> answer you wanted.
>
> It is invented.

---

## Act 3 — 🔍 So go and check

We are not going to *assert* that the answer is wrong. We have Hill Valley
County's actual policy manual sitting in `data/policy/`. Let's read it.

In [5]:
policy = (data.POLICY_DIR / "parcel_split_and_combination.md").read_text(encoding="utf-8")

# Pull out the minimum lot size table, straight from the office's own document.
for line in policy.splitlines():
    if "Minimum lot area" in line or "R-1 Single Family" in line:
        print(line.strip())

| Zoning district | Minimum lot area | Typical neighborhoods |
| R-1 Single Family | 6,000 sq ft | Hilldale, Lyon Estates, Courthouse Square |


In [6]:
viz.show_verification(
    claim=expert_answer,
    truth="6,000 sq ft per resulting parcel (R-1 Single Family).",
    source="data/policy/parcel_split_and_combination.md — Policy 4.3",
    question=QUESTION,
    consequence=(
        "This is not trivia. Biff Tannen's parcel at 1955 Hilldale Ct is "
        "15,800 sq ft, so a split produces two 7,900 sq ft lots. Under the real "
        "6,000 sq ft rule that <b>passes</b>. Under a 0.25 acre (10,890 sq ft) "
        "rule it <b>fails</b>. A wrong number here does not just misinform — "
        "it <b>reverses the decision</b>. Section 10 asks this exact question."
    ),
)

Let's make that concrete with the real parcel, out of the real database.

In [7]:
with data.cama() as con:
    parcel = con.execute(
        "SELECT * FROM parcels WHERE parcel_id = ?", ("04-127-3891",)
    ).fetchone()

lot = parcel["lot_size_sqft"]
half = lot // 2

print(f"Parcel 04-127-3891 — {parcel['situs_address']}")
print(f"  owner    : {parcel['owner_name']}")
print(f"  lot size : {lot:,} sq ft   ->  two lots of {half:,} sq ft")
print()
print(f"  under the REAL rule      (6,000 sq ft) : "
      f"{'SPLIT ALLOWED' if half >= 6_000 else 'DENIED'}")
print(f"  under the INVENTED rule (10,890 sq ft) : "
      f"{'SPLIT ALLOWED' if half >= 10_890 else 'DENIED'}")

Parcel 04-127-3891 — 1955 Hilldale Ct
  owner    : Biff Tannen
  lot size : 15,800 sq ft   ->  two lots of 7,900 sq ft

  under the REAL rule      (6,000 sq ft) : SPLIT ALLOWED
  under the INVENTED rule (10,890 sq ft) : DENIED


> ### 🎯 Same parcel. Same question. Opposite answers.
>
> One of those came from a document. The other came from a model that had never
> seen the document and produced the most plausible-looking text it could.

---

## 🎬 That's Section 4

The model is not lying to you. Lying requires knowing the truth.

It has **no clock**, **no internet**, and **no connection to your CAMA system**.
It was trained once, on text that existed before it was built, and then frozen.
When you ask it something outside that, it does the only thing it was ever
built to do: **predict a plausible next word.**

> ### 💡 A plausible answer and a correct answer look identical from the outside.
>
> That is the entire problem. You cannot tell by reading it. You could only tell
> because we had the county's real records sitting next to it.

Three things to carry forward:

1. 🔋 **It is blind, not broken.** Refusing honestly and inventing confidently
   are the same underlying limitation wearing different clothes.
2. ⚠️ **The persona that makes it useful is the persona that makes it invent.**
   Nobody deploys the bland prompt. Every office writes the confident one.
3. 📄 **Everything it got wrong, we had on disk.** The lot size was in a policy
   document. The parcel was in a database. It just could not reach them.

> ### ⏭️ **Next: Section 5 — Introducing Mr. Fusion.**
> If the model could *ask* for the policy document and *look up* the parcel,
> none of this would have happened. That is an **agent**, and it is the rest of
> this talk.

---

---

# 🔬 EXPLORE ON YOUR OWN

---

## 5. 🔍 The dangerous one — invented comparable sales

Ask for comps. This is the failure that would actually make it into a report.

In [8]:
comps_q = ("Give me three comparable sales near 1955 Hilldale Ct in Hill Valley "
           "County, with addresses and sale prices.")

comps_answer = ask(EXPERT, comps_q, max_tokens=170)
print(comps_answer)

Based on the Hill Valley County Assessor's records and recent sales history, here are three comparable sales for a property at **1955 Hilldale Ct**:

1.  **1952 Hilldale Ct** – Sold for **$485,000** (Closed: 14 days ago). This is a direct neighbor with identical lot dimensions and a similar two-story ranch style.
2.  **1948 Hilldale Ct** – Sold for **$472,000** (Closed: 3 weeks ago). Located on the same cul-de-sac, this home features the same age (1998) and square footage as the subject.
3.  **2001 Oakwood Dr** – Sold for **$495,0


In [9]:
# Now go to the Clerk of Courts and look. Comparables are selected by
# NEIGHBOURHOOD, not by street name -- 1955 Hilldale Ct is in Hilldale.
with data.cama() as con:
    hood = con.execute(
        "SELECT parcel_id, situs_address FROM parcels "
        "WHERE neighborhood_code = 'HILL' AND property_class IN ('R01','R02')"
    ).fetchall()

addresses = {r["parcel_id"]: r["situs_address"] for r in hood}
marks = ",".join("?" for _ in addresses)

with data.clerk() as con:
    sales = con.execute(
        f"SELECT parcel_id, recorded_date, sale_price FROM sales "
        f"WHERE parcel_id IN ({marks}) AND arms_length = 1 "
        f"ORDER BY recorded_date DESC", list(addresses)
    ).fetchall()

print("The three most recent QUALIFIED residential sales in Hilldale:\n")
for r in sales[:3]:
    print(f"  {addresses[r['parcel_id']]:<26} ${r['sale_price']:>9,}  {r['recorded_date']}")

# Do the addresses it gave us exist at all?
with data.cama() as con:
    on_street = {r["situs_address"] for r in con.execute(
        "SELECT situs_address FROM parcels WHERE situs_address LIKE '%Hilldale Ct'"
    )}

print(f"\nParcels addressed on Hilldale Ct countywide: {len(on_street)}")
for candidate in ("1952 Hilldale Ct", "1948 Hilldale Ct", "1955 Hilldale Ct"):
    print(f"  {candidate:<20} {'EXISTS' if candidate in on_street else 'DOES NOT EXIST'}")

The three most recent QUALIFIED residential sales in Hilldale:

  1797 Baines Way            $  266,000  2025-12-30
  2541 Hilldale Ct           $  374,000  2025-12-28
  2483 Peabody Farm Rd       $  292,000  2025-12-25

Parcels addressed on Hilldale Ct countywide: 84
  1952 Hilldale Ct     DOES NOT EXIST
  1948 Hilldale Ct     DOES NOT EXIST
  1955 Hilldale Ct     EXISTS


> ⚠️ Addresses, prices, and closing dates, often prefaced with *"based on the
> Assessor's records."* It never touched the Assessor's records. There is no
> flag on the output that says which parts were retrieved and which were
> invented, **because none of it was retrieved.**

---

## 6. 📊 Ask it about something everyone here knows cold

PRD is bread-and-butter to an assessment office. That makes it a perfect test,
because **you can only catch a hallucination in a domain you already know.**

In [10]:
print(ask(EXPERT,
          "In a ratio study, what does PRD stand for and what does a PRD of "
          "1.041 tell you?", max_tokens=150))

In the context of a ratio study within the Hill Valley County Assessor's Office, **PRD** stands for **Price Ratio Deviation**.

A PRD of **1.041** indicates that the assessed value of the subject property is **4.1% lower** than the sales ratio derived from comparable sales. Specifically, it means the sales ratio (Sales Price / Assessed Value) for the subject property is 1.041. This suggests the property is being assessed at a value 4.1% below its market value relative to the sales data, which typically warrants a downward adjustment to the assessed value to align with market conditions.


The correct answer, from `data/policy/ratio_study_standards.md`:

> **PRD is the Price Related Differential** — mean ratio ÷ (total assessed ÷
> total sale price). The IAAO standard is **0.98 to 1.03**. At **1.041** the
> stratum is **regressive**: lower-value properties are assessed at a higher
> percentage of their worth than higher-value ones.

Run the cell a few times on the local model and you may see it expand the
acronym **differently each time** — "Property Revenue Distribution", "Property
Reference Date". Neither is a real term.

---

## 7. ⚠️ The part that makes this genuinely hard

You might reasonably conclude: *fine, I will just never trust it.*

The problem is that **the failure is not reliable.** Watch the same expert
persona correctly refuse a question it has every reason to invent an answer to.

In [11]:
print(ask(EXPERT,
          "Summarize the March 2025 sale of parcel 04-127-3891 to Biff Tannen, "
          "including the sale price and the prior owner."))

There is no record of a March 2025 sale for parcel 04-127-3891 to Biff Tannen in the Hill Valley County Assessor's Office. This parcel does not exist in our current database, and Biff Tannen is not a recognized buyer in our sales history. Please verify the parcel number or the transaction date.


> ### 🤔 That sale is real — it is in `clerk.db` — and the model correctly said
> > it could not confirm it.
>
> So: it invents a zoning ordinance it was never told, and refuses to describe a
> sale that actually happened. **You cannot tell in advance which you will get,
> and the confident wrong answer looks exactly like the confident right one.**
>
> If it failed every time, you could plan around it. That would be *easier*.

---

## 8. 🕐 What year does it think it is?

The training cutoff is not an abstraction. Ask it directly.

In [12]:
print(ask(PLAIN, "What is today's date, and what is the most recent event you "
                 "have reliable knowledge of?"))

Hello! As an AI, I don't have access to real-time data, so I cannot tell you today's specific date. However, I can help you find that information if you'd like!

Regarding the most recent event I have reliable knowledge of, my training data is current up to my last update. If you're asking about a specific topic or event, feel free to ask, and I'll do my best to provide accurate information based on what I've been trained on!


Our Hill Valley County data runs through **2025**. Whatever the model just
said about its own knowledge is the gap — and that gap only ever grows after
the model ships.

---

<div align="center">

### 🕰️ *"Roads? Where we're going, we don't need roads."* 🕰️
### *…but we are going to need some tools.*

🚗💨 🔋 🏘️

</div>